# **Music Genre Classifier**

# Phase 0: Getting resources

In [1]:
# libraries needed.
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.offline import iplot, plot
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [5]:
# getting the dataset
read_data=pd.read_csv(r'D:\Music Genre Classification\dataset.csv', encoding='latin1')

In [19]:
# exploring the dataset
read_data.head()

,artist,song,duration_ms,explicit,year,popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,liveness,tempo,genre
0,Britney Spears,Oops!...I Did It Again,-711160.0,FALSE,2000.0,77.0,0.751,0.834,1,-5.444,0.0,0.0437,0.3000,0.3550,95.053,pop
1,blink-182,All The Small Things,167066.0,FALSE,1999.0,79.0,0.434,0.897,9,-4.918,1.0,0.0488,0.0103,0.6120,148.726,"rock, pop"
2,Faith Hill,Breathe,250546.0,FALSE,1999.0,66.0,0.529,0.496,7,-9.007,2.0,0.0290,0.1730,0.2510,136.859,"pop, country"
3,Bon Jovi,It's My Life,224493.0,FALSE,2030.0,78.0,0.551,0.913,0,-4.063,0.0,0.0466,0.0263,0.3470,119.992,"rock, metal"
4,*NSYNC,Bye Bye Bye,200560.0,FALSE,2000.0,645.0,0.614,0.928,8,-4.806,0.0,0.0516,0.0408,0.0845,172.656,pop


In [21]:
read_data.tail()

,artist,song,duration_ms,explicit,year,popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,liveness,tempo,genre
1995,Jonas Brothers,Sucker,181026.0,FALSE,2019.0,79.0,NaN,0.734,1,-5.065,0.0,0.0588,0.0427,0.1060,137.958,pop
1996,Taylor Swift,Cruel Summer,178426.0,FALSE,2019.0,78.0,0.552,0.702,9,-5.707,1.0,0.1570,0.1170,0.1050,169.994,pop
1997,Blanco Brown,The Git Up,200593.0,FALSE,2019.0,69.0,0.847,0.678,9,-8.635,6.0,0.1090,0.0669,0.2740,97.984,"hip hop, country"
1998,Sam Smith,Dancing With A Stranger (with Normani),171029.0,FALSE,2019.0,75.0,0.741,0.520,8,-7.513,1.0,0.0656,0.4500,0.2220,102.998,pop
1999,Post Malone,Circles,215280.0,FALSE,2019.0,85.0,0.695,90.762,0,-3.497,NaN,0.0395,0.1920,0.0863,120.042,hip hop


In [23]:
read_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   artist        2000 non-null   object 
 1   song          1995 non-null   object 
 2   duration_ms   1997 non-null   float64
 3   explicit      1997 non-null   object 
 4   year          1995 non-null   float64
 5   popularity    1996 non-null   float64
 6   danceability  1995 non-null   float64
 7   energy        1996 non-null   float64
 8   key           2000 non-null   int64  
 9   loudness      2000 non-null   float64
 10  mode          1997 non-null   float64
 11  speechiness   2000 non-null   float64
 12  acousticness  2000 non-null   float64
 13  liveness      2000 non-null   float64
 14  tempo         2000 non-null   float64
 15  genre         1983 non-null   object 
dtypes: float64(11), int64(1), object(4)
memory usage: 250.1+ KB


In [25]:
read_data.describe()

,duration_ms,year,popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,liveness,tempo
count,1997.000000,1995.000000,1996.000000,1995.000000,1996.000000,2000.000000,2000.000000,1997.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,226393.946420,2008.114286,60.699900,0.674769,0.843518,5.382500,-5.512434,0.645468,0.103568,0.128955,0.181216,120.122558
std,53049.016957,175.484156,27.869003,0.214446,2.976706,3.613962,1.933482,2.598075,0.096159,0.173346,0.140669,26.967112
min,-711160.000000,-2001.000000,0.000000,0.129000,0.054900,0.000000,-20.514000,0.000000,0.023200,0.000019,0.021500,60.019000
25%,203360.000000,2004.000000,56.000000,0.581000,0.623750,2.000000,-6.490250,0.000000,0.039600,0.014000,0.088100,98.985750
50%,223111.000000,2010.000000,66.000000,0.677000,0.737500,6.000000,-5.285000,1.000000,0.059850,0.055700,0.124000,120.021500
75%,247906.000000,2015.000000,73.000000,0.765000,0.840250,8.000000,-4.167750,1.000000,0.129000,0.176250,0.241000,134.265500
max,484146.000000,8000.000000,645.000000,5.607000,90.785000,11.000000,-0.276000,90.000000,0.576000,0.976000,0.853000,210.851000


In [27]:
read_data.isna().sum()

artist           0
song             5
duration_ms      3
explicit         3
year             5
popularity       4
danceability     5
energy           4
key              0
loudness         0
mode             3
speechiness      0
acousticness     0
liveness         0
tempo            0
genre           17
dtype: int64

In [29]:
read_data.duplicated().sum()

53

In [31]:
read_data.columns

Index(['artist', 'song', 'duration_ms', 'explicit', 'year', 'popularity',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'liveness', 'tempo', 'genre'],
      dtype='object')

# Phase 1: Data Cleaning

# Phase 2: EDA

# Phase 3: Feature Engineering

# Phase 4: Model Selection and Training

# Phase 5: Model Evaluation 